# Healthcare CVE Data Profile

## Questions

- How many valid CVE records are in the file?

> 1,515 records, and `CVE_ID` appears unique across all 1,515.

- Which fields need cleaning or type conversion?

> `Published` and `Last_Modified` should be converted to date fields.
>
> `CVSS_Score` should be converted to a numeric field.
>
> `Published_Year` should be derived to support annual analysis.
>
> Missing values are present in `Severity`, `CVSS_Score`, and `Attack_Vector` (18 each), and in `Weakness` (1).

- What are the main severity, attack-vector, keyword, and CWE distributions?

> `MEDIUM` is the largest severity bucket (720), `NETWORK` is the dominant attack vector (1,279), `hospital` is the largest keyword/domain (462), and `CWE-89` is the most common weakness (308), followed by `CWE-79` (247).

- Which metric definitions should the dashboard use?

> A few core metrics: total CVEs, critical CVEs, high/critical rate, network vector rate, yearly CVE volume, yearly high/critical network count, and top weakness/domain rankings.

## Notes

- `Severity`: how serious the vulnerability is, using labels like `LOW`, `MEDIUM`, `HIGH`, and `CRITICAL`.
- `CVSS_Score`: the numeric version of severity on a 0-10 scale.
- `Attack_Vector`: how the attacker reaches the vulnerable system.
- `NETWORK`: the issue can be exploited over a network, so the attacker does not need physical or local access first.
- `Weakness`: the type of coding or security problem behind the CVE.
- `CWE`: the standard label used for that weakness type, such as SQL injection or cross-site scripting.
- `Keyword`: the healthcare-related domain or topic that matched the CVE, such as `hospital`, `patient`, or `OpenEMR`.
- `Published_Year`: the year the CVE was published, used for annual trend analysis.

In [ ]:
# The sys.path lines make the import work
# whether the notebook server starts at the repo root or inside notebooks/.
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from analysis import (  # noqa: E402
    build_annual_trend,
    build_attack_vector_distribution,
    build_cvss_bands,
    build_domain_ranking,
    build_domain_severity_matrix,
    build_kpis,
    build_severity_distribution,
    build_triage_queue,
    build_weakness_ranking,
    clean_records,
    export_rows,
    headline_metrics,
    load_records,
)

DATA_PATH = repo_root / "data" / "raw" / "healthcare_cybersecurity_10k.csv"
df = load_records(DATA_PATH)
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isna().sum().sort_values(ascending=False)

## Basic Cleaning

In [ ]:
# errors="coerce" turns invalid values into missing ones instead of failing.
# clean_records performs exactly the four contract transformations and drops
# no rows, so every share keeps the full dataset as its denominator.
clean = clean_records(df)
clean.head()

In [ ]:
summary = {
    "records": len(clean),
    "unique_cves": clean["CVE_ID"].nunique(),
    "min_published": clean["Published"].min(),
    "max_published": clean["Published"].max(),
    "avg_cvss": clean["CVSS_Score"].mean(),
}
summary

## Distribution Checks

In [ ]:
clean["Severity"].value_counts(dropna=False)

In [ ]:
clean["Attack_Vector"].value_counts(dropna=False)

In [ ]:
clean["Weakness"].value_counts(dropna=False).head(15)

In [ ]:
clean["Keyword"].value_counts(dropna=False).head(15)

## First-Pass Metrics

In [ ]:
metrics = headline_metrics(clean)
metrics

## Dashboard Metric Definitions

All metric construction lives in `analysis/metrics.py`, so the notebook only
calls it and explains the results.

In [ ]:
metric_definitions = pd.DataFrame(
    [
        {
            "metric": "total_cves",
            "definition": "Count of parsed CVE records.",
            "why_it_matters": "Shows the size of the healthcare vulnerability set we are analyzing.",
        },
        {
            "metric": "critical_cves",
            "definition": "Count of records where Severity is CRITICAL.",
            "why_it_matters": "Shows how many vulnerabilities fall into the highest severity label.",
        },
        {
            "metric": "high_critical_rate",
            "definition": "Share of records where Severity is HIGH or CRITICAL.",
            "why_it_matters": "Shows how much of the dataset belongs to the more urgent severity tiers.",
        },
        {
            "metric": "network_vector_rate",
            "definition": "Share of records where Attack_Vector is NETWORK.",
            "why_it_matters": "Shows how much of the dataset may be reachable over a network, which is often more operationally urgent.",
        },
        {
            "metric": "annual_total_cves",
            "definition": "Count of CVEs by Published_Year.",
            "why_it_matters": "Shows how yearly vulnerability volume changes over time.",
        },
        {
            "metric": "annual_high_critical_network_cves",
            "definition": "Count of CVEs by year where Severity is HIGH or CRITICAL and Attack_Vector is NETWORK.",
            "why_it_matters": "Shows whether the most urgent network-exposed subset is rising along with total volume.",
        },
        {
            "metric": "severity_count",
            "definition": "Count of records by Severity label.",
            "why_it_matters": "Supports severity distribution views and headline risk summaries.",
        },
        {
            "metric": "attack_vector_count",
            "definition": "Count of records by Attack_Vector label.",
            "why_it_matters": "Supports exposure analysis by how an attacker can reach the vulnerable system.",
        },
        {
            "metric": "weakness_count",
            "definition": "Count of records by Weakness or CWE label.",
            "why_it_matters": "Turns individual CVEs into recurring prevention themes such as SQL injection or cross-site scripting.",
        },
        {
            "metric": "keyword_count",
            "definition": "Count of records by healthcare-related Keyword or domain.",
            "why_it_matters": "Shows which healthcare domains appear most often in the dataset.",
        },
    ]
)

metric_definitions

### Metric Notes

- `high_critical_rate` uses the current full dataset as the denominator, including rows with `N/A` values.
- `network_vector_rate` also uses the full dataset as the denominator.
- `NETWORK` is especially important because it suggests remote reachability.
- Weakness rankings should keep `NVD-CWE-noinfo` visible because it highlights incomplete classification.
- These are decision-support metrics, not predictive models.

In [ ]:
annual_output = build_annual_trend(clean)
annual_output

## Annual And Distribution Outputs

In [ ]:
plt.figure(figsize=(10, 4.5))
plt.plot(
    annual_output["Published_Year"],
    annual_output["total_cves"],
    marker="o",
    label="Total CVEs",
    color="#4c78a8",
)
plt.plot(
    annual_output["Published_Year"],
    annual_output["high_critical_network"],
    marker="o",
    label="High/Critical Network",
    color="#f58518",
)
plt.title("Annual CVE Volume and Network Exposure")
plt.xlabel("Published Year")
plt.ylabel("CVE Count")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
severity_distribution = build_severity_distribution(clean)
severity_distribution

In [ ]:
plt.figure(figsize=(8, 4.5))
sns.barplot(
    data=severity_distribution,
    x="count",
    y="severity",
    hue="severity",
    palette="crest",
    legend=False,
)
plt.title("Severity Distribution")
plt.xlabel("CVE Count")
plt.ylabel("Severity")
plt.tight_layout()
plt.show()

In [ ]:
attack_vector_distribution = build_attack_vector_distribution(clean)
attack_vector_distribution

In [ ]:
plt.figure(figsize=(8, 4.5))
sns.barplot(
    data=attack_vector_distribution,
    x="count",
    y="attack_vector",
    hue="attack_vector",
    palette="flare",
    legend=False,
)
plt.title("Attack Vector Distribution")
plt.xlabel("CVE Count")
plt.ylabel("Attack Vector")
plt.tight_layout()
plt.show()

In [ ]:
# inclusive="both" means each range includes both boundary values, such as 4.0 and 6.9.
# The bands do not partition the data: a null score and a 0.0 score fall into no band,
# so the shares deliberately do not sum to 1.
cvss_band_distribution = build_cvss_bands(clean)
cvss_band_distribution

In [ ]:
plt.figure(figsize=(8, 4.5))
sns.barplot(
    data=cvss_band_distribution,
    x="band",
    y="count",
    hue="band",
    palette=["#54a24b", "#9ecae9", "#f58518", "#e45756"],
    legend=False,
)
plt.title("CVSS Score Bands")
plt.xlabel("CVSS Band")
plt.ylabel("CVE Count")
plt.tight_layout()
plt.show()

## Ranking And Triage Outputs

In [ ]:
# NVD-CWE-noinfo stays visible because it highlights incomplete classification.
weakness_ranking = build_weakness_ranking(clean)
weakness_ranking

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(
    data=weakness_ranking,
    x="cve_count",
    y="Weakness",
    hue="Weakness",
    palette="mako",
    legend=False,
)
plt.title("Top Weakness Patterns")
plt.xlabel("CVE Count")
plt.ylabel("Weakness")
plt.tight_layout()
plt.show()

In [ ]:
# We keep raw counts for context, but sort by a simple priority score so high-severity and remotely reachable records rise faster.
# The priority score is a ranking heuristic for decision support, not a security score.
domain_ranking = build_domain_ranking(clean)
domain_ranking

In [ ]:
plt.figure(figsize=(10, 5.5))
sns.barplot(
    data=domain_ranking,
    x="priority_score",
    y="Keyword",
    hue="Keyword",
    palette="viridis",
    legend=False,
)
plt.title("Priority Domains")
plt.xlabel("Priority Score")
plt.ylabel("Keyword")
plt.tight_layout()
plt.show()

In [ ]:
# It depends on the ranking, so it is computed after domain_ranking.
domain_severity_matrix = build_domain_severity_matrix(clean, domain_ranking)
domain_severity_matrix

In [ ]:
plt.figure(figsize=(10, 5.5))
sns.heatmap(
    domain_severity_matrix,
    annot=True,
    fmt=".0f",
    cmap="YlOrRd",
    cbar_kws={"label": "CVE Count"},
)
plt.title("Domain Severity Matrix")
plt.xlabel("Severity")
plt.ylabel("Keyword")
plt.tight_layout()
plt.show()

In [ ]:
# Null Severity deliberately stays null in the output, unlike Keyword and Weakness.
triage_queue = build_triage_queue(clean)
triage_queue

## Dashboard Output Contract

In [ ]:
dashboard_kpis = build_kpis(clean)
dashboard_kpis

In [ ]:
contract_rows = [
    {
        "output_name": "dashboard_kpis",
        "panel": "Headline KPI row",
        "source_object": "dashboard_kpis",
        "key_columns": "metric, label, value, format",
    },
    {
        "output_name": "annual_trend",
        "panel": "Annual CVE Volume and Network Exposure",
        "source_object": "annual_output",
        "key_columns": "Published_Year, total_cves, high_critical_network",
    },
    {
        "output_name": "severity_distribution",
        "panel": "Severity Distribution",
        "source_object": "severity_distribution",
        "key_columns": "severity, count, share",
    },
    {
        "output_name": "attack_vector_distribution",
        "panel": "Attack Vector Distribution",
        "source_object": "attack_vector_distribution",
        "key_columns": "attack_vector, count, share",
    },
    {
        "output_name": "cvss_band_distribution",
        "panel": "CVSS Score Bands",
        "source_object": "cvss_band_distribution",
        "key_columns": "band, count, share",
    },
    {
        "output_name": "weakness_ranking",
        "panel": "Top Weakness Patterns",
        "source_object": "weakness_ranking",
        "key_columns": "Weakness, cve_count, critical_count, high_critical_count, network_count, avg_cvss",
    },
    {
        "output_name": "domain_ranking",
        "panel": "Priority Domains",
        "source_object": "domain_ranking",
        "key_columns": "Keyword, cve_count, critical_count, high_critical_count, network_count, avg_cvss, priority_score, network_rate",
    },
    {
        "output_name": "domain_severity_matrix",
        "panel": "Domain Severity Matrix",
        "source_object": "domain_severity_matrix",
        "key_columns": "Keyword index plus severity columns",
    },
    {
        "output_name": "triage_queue",
        "panel": "Triage Queue",
        "source_object": "triage_queue",
        "key_columns": "CVE_ID, Keyword, Severity, CVSS_Score, Attack_Vector, Weakness, Published, triage_score",
    },
]

dashboard_output_contract = pd.DataFrame(contract_rows)
dashboard_output_contract

In [ ]:
dashboard_output_bundle = {
    "dashboard_kpis": dashboard_kpis,
    "annual_trend": annual_output,
    "severity_distribution": severity_distribution,
    "attack_vector_distribution": attack_vector_distribution,
    "cvss_band_distribution": cvss_band_distribution,
    "weakness_ranking": weakness_ranking,
    "domain_ranking": domain_ranking,
    "domain_severity_matrix": domain_severity_matrix,
    "triage_queue": triage_queue,
}

list(dashboard_output_bundle.keys())

## Export Dashboard Data

In [ ]:
# export_rows converts every pandas null flavor to None, so strict JSON never
# emits NaN and a null date no longer crashes the serializer.
required_outputs = {
    "dashboard_kpis": "Run the Dashboard Output Contract section before exporting.",
    "annual_output": "Run the Annual And Distribution Outputs section before exporting.",
    "severity_distribution": "Run the Annual And Distribution Outputs section before exporting.",
    "attack_vector_distribution": "Run the Annual And Distribution Outputs section before exporting.",
    "cvss_band_distribution": "Run the Annual And Distribution Outputs section before exporting.",
    "weakness_ranking": "Run the Ranking And Triage Outputs section before exporting.",
    "domain_ranking": "Run the Ranking And Triage Outputs section before exporting.",
    "domain_severity_matrix": "Run the Ranking And Triage Outputs section before exporting.",
    "triage_queue": "Run the Ranking And Triage Outputs section before exporting.",
}

missing_outputs = [name for name in required_outputs if name not in globals()]
if missing_outputs:
    missing_list = ", ".join(missing_outputs)
    raise RuntimeError(
        "Missing notebook outputs: "
        + missing_list
        + ". Run the earlier analysis cells first, or use Run All before this export step."
    )

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

# Write the exported dashboard files into the repo-level data/processed folder.
output_dir = project_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

export_frames = {
    "kpis.json": dashboard_kpis,
    "annual_trend.json": annual_output,
    "severity_distribution.json": severity_distribution,
    "attack_vector_distribution.json": attack_vector_distribution,
    "cvss_bands.json": cvss_band_distribution,
    "weakness_ranking.json": weakness_ranking,
    "domain_ranking.json": domain_ranking,
    "domain_severity_matrix.json": domain_severity_matrix.reset_index(),
    "triage_queue.json": triage_queue,
}

written_files = []

for filename, frame in export_frames.items():
    rows = export_rows(frame)
    output_path = output_dir / filename
    output_path.write_text(json.dumps(rows, indent=2) + "\n", encoding="utf-8")
    written_files.append(filename)

written_files

## Decisions From Profiling

- Keep `N/A` values visible in severity and attack-vector counts.
- Include `N/A` rows in denominator-based rates unless we explicitly redefine the metric later.
- Keep `NVD-CWE-noinfo` in weakness rankings because it shows a classification gap.
- Use yearly CVE volume and yearly high/critical network count for the annual view.
- Use weighted priority score to sort domain rankings, while keeping raw counts visible for context.
- Use top weakness patterns, domain rankings, and a triage queue to connect the dataset to action.